In [1]:
import numpy as np
import numpy as np
from scipy.stats import norm, ncx2
from scipy.optimize import brentq

## Nelson-Siegel-Svensson (NSS) Model - Complete Implementation

This implementation calculates three key metrics from the NSS model:
1. **Zero-Coupon Yields (zcy)**: The yield on a zero-coupon bond at maturity T
2. **Discount Factors (P)**: The present value of $1 received at maturity T
3. **Instantaneous Forward Rates (f)**: The forward rates implied by the yield curve

In [2]:
def calculate_nss(maturities, a, b, c, d, tau, theta):
    """
    Parameters:
    -----------
    maturities : list, array, or float
        Time to maturity T (in years). Can be a single value or array of values.
    a : float
        Long-term level parameter
    b : float
        Short-term factor loading
    c : float
        Medium-term factor loading (hump)
    d : float
        Additional medium-term factor loading
    tau : float
        First decay parameter (controls curvature)
    theta : float
        Second decay parameter (controls steepness of hump)
    
    Returns:
    --------
    dict : Dictionary containing:
        - 'maturities': Input maturities
        - 'zero_coupon_yield': Zero-coupon yield at each maturity
        - 'discount_factor': Discount factor P(0,T) at each maturity
        - 'forward_rate': Instantaneous forward rate at each maturity
    """
    
    # Convert to numpy array for vectorized operations
    T = np.asarray(maturities, dtype=float)
    is_scalar = T.ndim == 0
    if is_scalar:
        T = np.array([T])
    
    # Initialize output arrays
    zcy = np.zeros_like(T)
    discount_factor = np.zeros_like(T)
    forward_rate = np.zeros_like(T)
    
    for i, t in enumerate(T):
        if t == 0:
            # Edge case: at T=0, yield is simply a + b
            zcy[i] = a + b
        else:
            # Zero-Coupon Yield formula
            # zcy(T) = a + b*(T/τ)/(1-exp(-T/τ)) + c*((T/τ)/(1-exp(-T/τ)) - exp(-T/τ)) 
            #          + d*((T/θ)/(1-exp(-T/θ)) - exp(-T/θ))
            
            exp_neg_t_tau = np.exp(-t / tau)
            exp_neg_t_theta = np.exp(-t / theta)
            
            term1 = (1 - exp_neg_t_tau) / (t / tau)
            term2 = (1 - exp_neg_t_theta) / (t / theta)
            
            zcy[i] = (a + 
                     b * term1 + 
                     c * (term1 - exp_neg_t_tau) + 
                     d * (term2 - exp_neg_t_theta))
        
        # Discount Factor formula
        # P(0,T) = exp(-zcy(T) * T)
        discount_factor[i] = np.exp(-zcy[i] * t)
        
        # Instantaneous Forward Rate formula
        # f(T) = a + b*exp(-T/τ) + c*(T/τ)*exp(-T/τ) + d*(T/θ)*exp(-T/θ)
        forward_rate[i] = (a + 
                          b * np.exp(-t / tau) + 
                          c * (t / tau) * np.exp(-t / tau) + 
                          d * (t / theta) * np.exp(-t / theta))
    
    # Return scalar values if input was scalar
    if is_scalar:
        return {
            'maturities': maturities,
            'zero_coupon_yield': zcy[0],
            'discount_factor': discount_factor[0],
            'forward_rate': forward_rate[0]
        }
    else:
        return {
            'maturities': T,
            'zero_coupon_yield': zcy,
            'discount_factor': discount_factor,
            'forward_rate': forward_rate
        }

## Example: Plug in Your Parameters

In [ ]:
# ============================================
# EASY TO USE: Plug in your parameter values here
# ============================================

# NSS Model Parameters
a = 0.04          # Long-term level
b = -0.01         # Short-term factor
c = 0.02          # Medium-term factor (hump)
d = -0.005        # Additional medium-term factor
tau = 2.0         # First decay parameter
theta = 0.5       # Second decay parameter

# List of maturities (in years)
maturities = [0.5, 1, 2, 5, 10]

# Calculate NSS model values
results = calculate_nss(maturities, a, b, c, d, tau, theta)

# Display results
print("=" * 80)
print("Nelson-Siegel-Svensson (NSS) Model Results")
print("=" * 80)
print(f"\nNSS Parameters: a={a}, b={b}, c={c}, d={d}, tau={tau}, theta={theta}\n")

print(f"{'Maturity (T)':<15} {'Zero-Coupon Yield':<20} {'Discount Factor':<20} {'Forward Rate':<20}")
print("-" * 80)

for i, t in enumerate(results['maturities']):
    zcy = results['zero_coupon_yield'][i] * 100  # Convert to percentage
    df = results['discount_factor'][i]
    fr = results['forward_rate'][i] * 100  # Convert to percentage
    print(f"{t:<15.2f} {zcy:<19.4f}% {df:<19.6f} {fr:<19.4f}%")

print("\n" + "=" * 80)

Nelson-Siegel-Svensson (NSS) Model Results

NSS Parameters: a=0.04, b=-0.01, c=0.02, d=-0.005, tau=2.0, theta=0.5

Maturity (T)    Zero-Coupon Yield    Discount Factor      Forward Rate        
--------------------------------------------------------------------------------
0.50            3.1951             % 0.984152            3.4267             %
1.00            3.4254             % 0.966326            3.8647             %
2.00            3.7828             % 0.927135            4.3312             %
5.00            4.1530             % 0.812491            4.3281             %
10.00           4.1602             % 0.659669            4.0606             %



: 

**Vasicek Model**

In [ ]:
# 1. VASICEK MODEL

def vasicek_params(kappa, theta, sigma):
    """Returns A(tau) and B(tau) components for Vasicek."""
    def B(tau):
        return (1 - np.exp(-kappa * tau)) / kappa

    def A(tau):
        B_val = B(tau)
        term1 = (theta - (sigma**2) / (2 * kappa**2)) * (B_val - tau)
        term2 = (sigma**2) / (4 * kappa) * (B_val**2)
        return np.exp(term1 - term2)
    return A, B

def price_vasicek_zcb(r0, kappa, theta, sigma, T):
    """Price of a Zero-Coupon Bond (ZCB) under Vasicek."""
    # T is time to maturity (T - t)
    A, B = vasicek_params(kappa, theta, sigma)
    return A(T) * np.exp(-B(T) * r0)

def price_vasicek_option_zcb(r0, kappa, theta, sigma, T_bond, T_option, K, option_type='call'):
    """European Option on ZCB under Vasicek (Eq 3.12)."""
    # T_bond: Maturity of the underlying bond (measured from today)
    # T_option: Maturity of the option (measured from today)
    
    P_bond = price_vasicek_zcb(r0, kappa, theta, sigma, T_bond)
    P_option = price_vasicek_zcb(r0, kappa, theta, sigma, T_option)
    
    # Volatility of the bond price (sigma_P) [4]
    # Note: Text defines sigma_p based on (T - T_call)
    tau = T_bond - T_option
    sigma_p = (sigma / kappa) * (1 - np.exp(-kappa * tau)) * np.sqrt((1 - np.exp(-2 * kappa * T_option)) / (2 * kappa))
    
    # h calculation [4]
    h = (1 / sigma_p) * np.log(P_bond / (K * P_option)) + (sigma_p / 2)
    
    if option_type == 'call':
        return P_bond * norm.cdf(h) - K * P_option * norm.cdf(h - sigma_p)
    else: # Put (via Put-Call Parity logic or Eq 3.15)
        return K * P_option * norm.cdf(sigma_p - h) - P_bond * norm.cdf(-h)

**COX-INGERSOLL-ROSS (CIR) MODEL**

In [ ]:
def cir_params(kappa, theta, sigma):
    """Returns A(tau) and B(tau) components for CIR."""
    gamma = np.sqrt(kappa**2 + 2*sigma**2)
    
    def B(tau):
        num = 2 * (np.exp(gamma * tau) - 1)
        den = (kappa + gamma) * (np.exp(gamma * tau) - 1) + 2 * gamma
        return num / den

    def A(tau):
        num = 2 * gamma * np.exp((kappa + gamma) * tau / 2)
        den = (kappa + gamma) * (np.exp(gamma * tau) - 1) + 2 * gamma
        exponent = 2 * kappa * theta / sigma**2
        return (num / den) ** exponent
    return A, B

def price_cir_zcb(r0, kappa, theta, sigma, T):
    """Price of ZCB under CIR."""
    A, B = cir_params(kappa, theta, sigma)
    return A(T) * np.exp(-B(T) * r0)

def price_cir_option_zcb(r0, kappa, theta, sigma, T_bond, T_option, K, option_type='call'):
    """European Option on ZCB under CIR (Eq 3.39)."""
    # Calculation of auxiliary variables [9]
    gamma = np.sqrt(kappa**2 + 2*sigma**2)
    A, B = cir_params(kappa, theta, sigma)
    
    # Bond parameters B(T-T_call) and A(T-T_call)
    tau_bond = T_bond - T_option 
    B_tau = B(tau_bond) 
    A_tau = A(tau_bond) 

    phi = 2 * gamma / (sigma**2 * (np.exp(gamma * T_option) - 1))
    psi = (kappa + gamma) / sigma**2
    xi = phi + psi + B_tau
    
    # Degrees of freedom and Non-centrality
    df = 4 * kappa * theta / sigma**2
    nc_param_1 = (2 * phi**2 * r0 * np.exp(gamma * T_option)) / xi
    nc_param_2 = (2 * phi**2 * r0 * np.exp(gamma * T_option)) / (phi + psi)
    
    # Critical values for Chi-Square
    # The formula involves ln(A/K)... check Eq 3.39 carefully
    val_1 = 2 * xi * (np.log(A_tau / K)) / B_tau # This derivation assumes K is price strike
    # Note: Sometimes K is Yield strike. The prompt implies Price strike (value of option).
    # Using explicit formulation from notes:
    # The text 3.39 has arguments inside chi-square. 
    # Let's map x to the text's arguments:
    
    x1 = 2 * xi * (np.log(A_tau/K)) / B_tau # Caution: If K is high, log is negative. 
    # However, for Chi2 CDF, x must be positive. 
    # The formula in notes relies on K being the strike price of the BOND.
    # The chi-square integral limit is often 2*rho*r*.
    
    # ALTERNATIVE: Explicit integration or using the "Psi, Phi" directly from notes.
    # Let's trust the formula structure provided in notes for call:
    
    P_bond = price_cir_zcb(r0, kappa, theta, sigma, T_bond)
    P_option = price_cir_zcb(r0, kappa, theta, sigma, T_option)
    
    # Limits for Chi-Square (Eq 3.39)
    # Arg 1 for first Chi2
    limit1 = 2 * xi * (np.log(A_tau/K)) / B_tau
    # Arg 1 for second Chi2
    limit2 = 2 * (phi + psi) * (np.log(A_tau/K)) / B_tau
    
    # Python's ncx2.cdf(x, df, nc)
    # The notes use complementary Chi-square or standard? 
    # Standard pricing usually is P * Chi2_1 - K * P_opt * Chi2_2
    
    chi2_1 = ncx2.cdf(limit1, df, nc_param_1)
    chi2_2 = ncx2.cdf(limit2, df, nc_param_2)
    
    if option_type == 'call':
        return P_bond * chi2_1 - K * P_option * chi2_2
    else:
        # Put-Call Parity: Put = Call - P_bond + K * P_option
        call_val = P_bond * chi2_1 - K * P_option * chi2_2
        return call_val - P_bond + K * P_option

**HULL-WHITE MODEL (Analytical)**

In [ ]:
def price_hw_zcb_market(r0, kappa, sigma, T, Pm_0_T, Pm_0_t=1.0, f_m_0_t=None):
    """
    Hull-White ZCB Price given Market Data.
    If t=0 (current valuation), Price is simply Market Price.
    If forward valuation (t > 0), we need the formula 3.41.
    """
    # Usually quizzes ask for price at t=0, which is just the market price input.
    # If they ask for P(t, T) given r_t:
    # We need market price P(0,T) and P(0,t).
    
    # Calculate B_hw (Same as Vasicek)
    B = (1 - np.exp(-kappa * T)) / kappa # Note: T here is actually (T-t) in the formula
    # But usually T is absolute maturity.
    # Let's assume input T is time-to-maturity for simplicity or strictly follow Eq 3.41
    # where arguments are (t, T).
    pass 
    # NOTE: For the quiz, if asked for HW price at t=0, it equals the observed market price.
    # If parameters given are just kappa/sigma/flat curve, it collapses to Vasicek with theta(t)

**COUPON BOND OPTION (Jamshidian)**

In [ ]:
def jamshidian_decomposition(r0, model_type, params, T_option, coupons, times, K_strike, option_type='call'):
    """
    Prices an option on a coupon bond using Jamshidian's decomposition.
    model_type: 'vasicek' or 'cir'
    params: dict of model parameters
    coupons: list of cash flows (including principal)
    times: list of times for cash flows (must be > T_option)
    K_strike: Strike price of the option on the COUPON BOND
    """
    
    # 1. Define function to value bond at option expiry (t = T_option) as function of r
    def bond_price_at_expiry(r):
        value = 0
        for c, t in zip(coupons, times):
            tau = t - T_option
            if model_type == 'vasicek':
                # P(T_opt, t)
                p = price_vasicek_zcb(r, params['kappa'], params['theta'], params['sigma'], tau)
            elif model_type == 'cir':
                p = price_cir_zcb(r, params['kappa'], params['theta'], params['sigma'], tau)
            value += c * p
        return value - K_strike

    # 2. Find critical r* (r_star) that makes Bond Price = Strike
    # We assume r is between -0.5 and 0.5 (safe bounds)
    r_star = brentq(bond_price_at_expiry, -0.1, 0.5)
    
    # 3. Calculate individual strikes (Ki) for each ZCB
    total_option_value = 0
    for c, t in zip(coupons, times):
        tau = t - T_option
        if model_type == 'vasicek':
            # Ki is the price of ZCB at r*
            Ki = price_vasicek_zcb(r_star, params['kappa'], params['theta'], params['sigma'], tau)
            # Value option on this ZCB
            # Strike for the option on ZCB is Ki
            # Nominal is c
            # Formula expects strike K. Here strike is Ki.
            opt_val = price_vasicek_option_zcb(r0, params['kappa'], params['theta'], params['sigma'], 
                                               t, T_option, Ki, option_type)
        elif model_type == 'cir':
            Ki = price_cir_zcb(r_star, params['kappa'], params['theta'], params['sigma'], tau)
            opt_val = price_cir_option_zcb(r0, params['kappa'], params['theta'], params['sigma'], 
                                           t, T_option, Ki, option_type)
            
        total_option_value += c * opt_val
        
    return total_option_value

**CAPS AND FLOORS**

In [ ]:
def price_cap_floor(r0, model_type, params, notional, K_rate, tenor, freq, option_type='cap'):
    """
    Prices Cap or Floor as sum of ZCB options.
    tenor: total years (e.g. 1 year)
    freq: payment frequency (e.g. 0.25 for quarterly)
    K_rate: Cap/Floor rate (e.g. 0.06)
    """
    
    # Generate reset/payment dates
    # First reset usually at t=0 or t=freq. Assuming starts now, payments at freq, 2*freq...
    # Caplet j pays at t_j, resets at t_{j-1}
    # Important: First period usually not capped if rate is already set. 
    # Standard Cap: Sum of options from j=2 to n (if first rate known) or j=1 (if forward start).
    # Assuming standard spot start: First caplet covers [t1, t2], pays at t2.
    
    num_payments = int(tenor / freq)
    total_value = 0
    
    # Nominal for the Put/Call on ZCB calculation:
    # M' = M * (1 + K * tau)
    M_prime = notional * (1 + K_rate * freq)
    
    # Strike for the Put/Call on ZCB:
    # K' = 1 / (1 + K * tau)
    K_prime = 1.0 / (1 + K_rate * freq)
    
    # Loop through caplets
    for i in range(1, num_payments + 1):
        t_pay = i * freq      # Maturity of the ZCB (T_bond)
        t_reset = (i-1) * freq # Maturity of the Option (T_option)
        
        # If t_reset is 0, the rate is fixed (known r0). 
        # Usually Caplets start for reset > 0. Let's assume we price the whole stream.
        # But if reset is 0, there is no volatility, it's just intrinsic value.
        # Most models/exercises start option pricing at first future reset.
        if t_reset == 0:
            continue 

        if model_type == 'vasicek':
            # Caplet = Put on ZCB
            # Floorlet = Call on ZCB
            # Mapping: Cap -> Put, Floor -> Call [12]
            
            zcb_opt_type = 'put' if option_type == 'cap' else 'call'
            
            val = price_vasicek_option_zcb(r0, params['kappa'], params['theta'], params['sigma'], 
                                           t_pay, t_reset, K_prime, zcb_opt_type)
            
        elif model_type == 'cir':
            zcb_opt_type = 'put' if option_type == 'cap' else 'call'
            val = price_cir_option_zcb(r0, params['kappa'], params['theta'], params['sigma'], 
                                       t_pay, t_reset, K_prime, zcb_opt_type)
            
        total_value += M_prime * val
        
    return total_value

**EXECUTION BLOCK**

In [ ]:
if __name__ == "__main__":
    # Example Parameters (Replace with Quiz Values)
    r0_val = 0.05
    kappa_val = 0.15
    theta_val = 0.05
    sigma_val = 0.01
    
    # 1. ZCB Price (Vasicek)
    print("Vasicek ZCB (T=5):", price_vasicek_zcb(r0_val, kappa_val, theta_val, sigma_val, 5))
    
    # 2. Option on ZCB (European Call)
    # Option expires in 1 year, Bond matures in 5 years, Strike 0.95
    print("Vasicek Call on ZCB:", price_vasicek_option_zcb(r0_val, kappa_val, theta_val, sigma_val, 
                                                           T_bond=5, T_option=1, K=0.95, option_type='call'))
                                                           
    # 3. Coupon Bond Option (Jamshidian)
    # Bond: 3 years, semi-annual coupons (0.5, 1.0 ... 3.0), Coupon Rate 4%
    # Option: Expire in 1 year. Strike 0.98.
    # Note: Cash flows occur at 1.5, 2.0, 2.5, 3.0 (since option is at 1.0)
    # Cash flows must be defined relative to today.
    times = [1.5, 2.0, 2.5, 3.0] 
    coupons = [13, 14] # $100 face value, 4% rate = $2 coupon
    
    params_v = {'kappa': kappa_val, 'theta': theta_val, 'sigma': sigma_val}
    print("Vasicek Coupon Bond Option:", jamshidian_decomposition(r0_val, 'vasicek', params_v, 
                                                                  T_option=1.0, coupons=coupons, times=times, 
                                                                  K_strike=98, option_type='call'))

    # 4. Cap/Floor
    # 1 year Cap, Quarterly payment, Strike 6%
    print("Vasicek Cap Value:", price_cap_floor(r0_val, 'vasicek', params_v, 
                                                notional=100, K_rate=0.06, tenor=1, freq=0.25, option_type='cap'))